In [2]:
!pip install duckdb pandas

In [3]:
import duckdb
import pandas as pd

# 1. Create an in-memory DuckDB connection
con = duckdb.connect()

# 2. Define your exact file paths
parquet_path = "data/yellow_tripdata_2026-03.parquet"
csv_path = "data/taxi_zone_lookup.csv"

# 3. Create Views pointing directly to your files
con.execute(f"CREATE VIEW trips AS SELECT * FROM '{parquet_path}'")
con.execute(f"CREATE VIEW zones AS SELECT * FROM '{csv_path}'")

print("✅ Data successfully connected and views created!")

✅ Data successfully connected and views created!


In [26]:
# Display columns related to money and distance
display(raw_data_profile[raw_data_profile['column_name'].isin(['trip_distance', 'total_amount', 'fare_amount', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee', 'passenger_count', 'VendorID', 'RatecodeID', 'extra', 'mta_tax'])])

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,7,4,1.8778934893816521,0.7248573288489323,2,2,2,3952451,0.00
3,passenger_count,BIGINT,0,8,9,1.2308229312971717,0.6330375072054865,1,1,1,3952451,23.93
4,trip_distance,DOUBLE,0.0,288381.68,3540,5.9836929464770074,581.0544867448335,1.016400005756196,1.82846895454698,3.714697788905559,3952451,0.00
5,RatecodeID,BIGINT,1,99,7,5.6354106142176335,20.591522788812032,1,1,1,3952451,23.93
10,fare_amount,DOUBLE,-990.0,1843.3,10519,21.327424967447843,18.6868917350294,9.956709457477048,15.557281451151763,26.804288097325795,3952451,0.00
11,extra,DOUBLE,-7.5,17.46,56,1.115754401509342,1.746227364647829,0.0,0.0,2.498884096701159,3952451,0.00
12,mta_tax,DOUBLE,-0.5,4.75,7,0.4885423120995048,0.09035171054149621,0.5,0.5,0.5,3952451,0.00
13,tip_amount,DOUBLE,-75.75,397.0,3777,2.8575176744755737,3.914010258674575,0.0,2.129334557213604,3.9523582797611927,3952451,0.00
14,tolls_amount,DOUBLE,-111.79,1400.0,1271,0.5283688374632909,2.285484460199622,0.0,0.0,0.0,3952451,0.00
15,improvement_surcharge,DOUBLE,-1.0,1.0,4,0.9521509311564333,0.23606851988482797,1.0,1.0,1.0,3952451,0.00


In [20]:
# 1. Inspect the Yellow Taxi Trip Data (The Fact Table)
print("🚕 --- YELLOW TAXI TRIPS (Parquet File) --- 🚕")
trips_columns_df = con.execute("DESCRIBE trips").df()
display(trips_columns_df)

print("\n")

# 2. Inspect the Taxi Zone Lookup (The Dimension Table)
print("🗺️ --- TAXI ZONE LOOKUP (CSV File) --- 🗺️")
zones_columns_df = con.execute("DESCRIBE zones").df()
display(zones_columns_df)

🚕 --- YELLOW TAXI TRIPS (Parquet File) --- 🚕


,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None




🗺️ --- TAXI ZONE LOOKUP (CSV File) --- 🗺️


,column_name,column_type,null,key,default,extra
0,LocationID,BIGINT,YES,None,None,None
1,Borough,VARCHAR,YES,None,None,None
2,Zone,VARCHAR,YES,None,None,None
3,service_zone,VARCHAR,YES,None,None,None


In [28]:
# Check for Datetime and Categorical Anomalies
validation_query = """
    SELECT 
        -- 1. Time Travel Checks
        MIN(tpep_pickup_datetime) AS earliest_pickup,
        MAX(tpep_pickup_datetime) AS latest_pickup,
        
        -- Check for impossible time (Dropoff happened BEFORE pickup)
        SUM(CASE WHEN tpep_dropoff_datetime < tpep_pickup_datetime THEN 1 ELSE 0 END) AS negative_duration_trips,
        
        -- 2. Categorical Checks (What weird values exist?)
        MIN(PULocationID) AS min_pickup_id,
        MAX(PULocationID) AS max_pickup_id,
        MIN(payment_type) AS min_payment_type,
        MAX(payment_type) AS max_payment_type
        
    FROM trips;
"""

print("🔍 Validating Dates and Categories...")
display(con.execute(validation_query).df())

# Let's also check the distinct values of the flag
flag_query = """
    SELECT store_and_fwd_flag, COUNT(*) as count 
    FROM trips 
    GROUP BY store_and_fwd_flag;
"""
print("\n🚩 Store and Forward Flag Values:")
display(con.execute(flag_query).df())

🔍 Validating Dates and Categories...


,earliest_pickup,latest_pickup,negative_duration_trips,min_pickup_id,max_pickup_id,min_payment_type,max_payment_type
0,2008-12-31 23:03:20,2026-04-01 00:06:25,2.0,1,265,0,4



🚩 Store and Forward Flag Values:


,store_and_fwd_flag,count
0,N,3001767
1,None,945748
2,Y,4936


In [40]:
import duckdb

con = duckdb.connect()

# Re-establish views
parquet_path = "data/yellow_tripdata_2026-03.parquet"
csv_path = "data/taxi_zone_lookup.csv"
con.execute(f"CREATE VIEW IF NOT EXISTS trips AS SELECT * FROM '{parquet_path}'")
con.execute(f"CREATE VIEW IF NOT EXISTS zones AS SELECT * FROM '{csv_path}'")

# The TRULY Final Master Query (With Boroughs included!)
master_clean_query = """
    CREATE OR REPLACE TABLE clean_ml_data AS
    SELECT 
        -- 1. Time Features 
        trip.tpep_pickup_datetime,
        trip.tpep_dropoff_datetime,
        EXTRACT(HOUR FROM trip.tpep_pickup_datetime) AS pickup_hour,
        EXTRACT(ISODOW FROM trip.tpep_pickup_datetime) AS day_of_week,
        
        -- 2. Spatial Features (Joined & Cleaned)
        trip.PULocationID,
        pz.Zone AS pickup_zone,
        pz.Borough AS pickup_borough,     -- < Added back!
        trip.DOLocationID,
        dz.Zone AS dropoff_zone,
        dz.Borough AS dropoff_borough,    -- < Added back!
        
        -- 3. Operational Features
        COALESCE(trip.passenger_count, 1) AS passenger_count,
        trip.trip_distance,
        
        -- 4. Financial Features
        trip.payment_type,
        trip.fare_amount,
        COALESCE(trip.tip_amount, 0) AS tip_amount,
        trip.total_amount
        
    FROM trips AS trip
    JOIN zones AS pz ON trip.PULocationID = pz.LocationID
    JOIN zones AS dz ON trip.DOLocationID = dz.LocationID
    
    WHERE 
        trip.fare_amount > 2.50
        AND trip.total_amount > 0
        AND trip.tip_amount >= 0
        AND trip.trip_distance > 0.1 
        AND trip.trip_distance < 100 
        AND pz.Borough != 'Unknown'
        AND dz.Borough != 'Unknown'
        AND trip.tpep_pickup_datetime >= '2026-03-01 00:00:00'
        AND trip.tpep_pickup_datetime < '2026-04-01 00:00:00'
        AND trip.tpep_dropoff_datetime > trip.tpep_pickup_datetime
        AND trip.payment_type IN (1, 2, 3, 4);
"""

# Execute and save
con.execute(master_clean_query)
con.execute("COPY clean_ml_data TO 'data/cleaned_taxi_data_for_ml.parquet' (FORMAT PARQUET)")
print("✅ Clean data updated with Boroughs and saved!")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Clean data updated with Boroughs and saved!


In [38]:
import duckdb

con = duckdb.connect()
clean_data = "'data/cleaned_taxi_data_for_ml.parquet'"

tip_check_query = f"""
    SELECT 
        payment_type,
        COUNT(*) AS total_trips,
        ROUND(AVG(tip_amount), 2) AS avg_tip,
        SUM(CASE WHEN tip_amount = 0 THEN 1 ELSE 0 END) AS zero_tip_count
    FROM {clean_data}
    GROUP BY payment_type
    ORDER BY payment_type;
"""
print(con.execute(tip_check_query).df())

   payment_type  total_trips  avg_tip  zero_tip_count
0             1      2534709     4.13        248503.0
1             2       328319     0.00        328303.0
2             3         6936     0.00          6935.0
3             4        14496     0.00         14494.0
